# Week 11 guided practice
CML3111. Follow the lecturer’s PowerPoint in this order. This notebook contains worked examples, not your graded upload.

Extract the student ZIP and keep its files together. For file-reading examples, upload the supplied CSV into the Colab runtime before running the file-reading example. From the file-reading lesson onward, worked table examples use the supplied CSV. Earlier syntax examples introduce Python before file access.


## Today’s learning
*PowerPoint slide 2*

Use pandas tables for this week’s data work. Explain the decisions and check the outputs.

Complete the issued function interfaces and written evidence using the demonstrated workflow.




## The supplied file
*PowerPoint slide 3*

Extract this week’s student ZIP. Upload engineering_data.csv into Colab using the Files panel, or keep it beside the notebook locally.

The examples below read this file. Run the cells in slide order. Keep the original CSV unchanged.




## Operations for the next example
*PowerPoint slide 4*

Import and short names: import pandas as pd loads pandas and gives it the short name pd. pd.read_csv refers to a function in that library.

Reading a CSV: pd.read_csv("practice_filename.csv") reads the named file into a DataFrame: a table with labelled columns and row labels. The file must exist in the working folder.

Counting dimensions: equipment_data.shape gives (number of rows, number of columns). shape is a property, so it has no parentheses.




## Operations for the next example
*PowerPoint slide 5*

Column data types: equipment_data.dtypes reports each column’s stored type. int64 means integers, float64 means decimal numbers, and object often contains text. Numeric blanks commonly appear as NaN.

A preview of rows: equipment_data.head() returns the first five rows. It leaves the table unchanged. A preview is not the entire dataset.




## Reading the code syntax
*PowerPoint slide 6*

= assigns a value to a name; == compares values. Parentheses pass inputs or continue an expression across lines. print displays a value; len counts entries. int and float convert to integer and decimal numeric values.




## Loading the supplied observations
*PowerPoint slide 7*



Compare the printed result with the explanation in the notebook.


In [ ]:
import pandas as pd

raw_data = pd.read_csv("engineering_data.csv")
print(raw_data.head())
print(raw_data.shape)
print(raw_data.dtypes)


**Example output** (timings and display formatting may vary)

```text
record_id equipment_id  load_pct  ambient_c  target
0          1           E1     59.15      31.71  76.847
1          2           E2     77.46      29.97  87.047
2          3           E3     69.60      19.87  71.256
3          4           E4     75.98      22.29     NaN
4          5           E5     57.62      23.40  67.034
(91, 5)
record_id         int64
equipment_id     object
load_pct        float64
ambient_c       float64
target          float64
dtype: object
```


## Rows used for modelling
*PowerPoint slide 8*

Keep the first occurrence of each record_id. Retain recorded targets and load_pct within 0–100. Then separate E1–E4 for training, E5 for validation and E6 for the final test_data.

Fill missing input values using training data only. Keep final-test targets out of model selection.




## Operations for the next example
*PowerPoint slide 9*

Reading chained operations: Read a chain from left to right. In equipment_data["target"].mean(), select the target Series first, then calculate its mean. Each step uses the previous step’s result.

A separate working table: equipment_data.copy() creates a working copy. Assigning columns on this copy preserves the source table, so you can compare before and after.

Finding recorded cells: notna() returns True where a value is present. Use it to retain observations with a recorded target before modelling.




## Operations for the next example
*PowerPoint slide 10*

An inclusive range: series.between(0, 100) is True for values from 0 through 100, including both limits. The resulting mask helps select valid rows.

Selecting columns: equipment_data["column"] returns one labelled column, a Series. equipment_data[["a", "b"]] returns a DataFrame containing those named columns in that order.

Selecting rows with a mask: A column comparison produces one True or False per row. equipment_data[mask] keeps the True rows. & combines masks: both conditions must hold. Parenthesise each comparison.




## Operations for the next example
*PowerPoint slide 11*

Keeping the first observation: equipment_data.drop_duplicates("record_id") returns rows with the first occurrence of each ID, preserving their order. It does not overwrite the source unless you assign the result.

Membership in a group list: series.isin(["E1", "E2"]) returns True for entries matching either listed label. equipment_data[that_mask] keeps all observations belonging to those groups.




## Prepare valid observations
*PowerPoint slide 12*

First keep the first record ID. Then ask two questions for each remaining row: is the target recorded, and is load between 0 and 100? Only a row passing both checks stays. Missing inputs are handled later using training data only.




## Preparing the supplied modelling rows — step 1
*PowerPoint slide 13*



Compare the printed result with the explanation in the notebook.


In [ ]:
modelling_data = raw_data.drop_duplicates("record_id").copy()
target_is_recorded = modelling_data["target"].notna()
load_is_valid = modelling_data["load_pct"].between(0, 100)
rows_to_keep = target_is_recorded & load_is_valid
print(target_is_recorded.head())
print(load_is_valid.head())
modelling_data = modelling_data[rows_to_keep].copy()
print(len(raw_data), len(modelling_data))


**Example output** (timings and display formatting may vary)

```text
0     True
1     True
2     True
3    False
4     True
Name: target, dtype: bool
0    True
1    True
2    True
3    True
4    True
Name: load_pct, dtype: bool
91 88
```


## Choose groups before choosing rows
*PowerPoint slide 14*

isin(list_of_labels) answers True when a row’s equipment ID appears in the list. Here E1–E4 supply training rows, E5 validation rows and E6 test rows. The groups stay separate so one equipment group cannot appear on both sides.




## Preparing the supplied modelling rows — step 2
*PowerPoint slide 15*



Compare the printed result with the explanation in the notebook.


In [ ]:
is_training_equipment = modelling_data["equipment_id"].isin(
    ["E1", "E2", "E3", "E4"]
)
is_validation_equipment = modelling_data["equipment_id"].isin(
    ["E5"]
)
is_test_equipment = modelling_data["equipment_id"].isin(["E6"])


**Example output** (timings and display formatting may vary)

```text
This cell stores values for the next example; it does not print a result.
```


## Preparing the supplied modelling rows — step 3
*PowerPoint slide 16*

training_data: rows used to learn model settings; only these rows may determine imputation values.

validation_data: separate rows used to compare models without fitting on them.




## Preparing the supplied modelling rows — step 3
*PowerPoint slide 17*



Compare the printed result with the explanation in the notebook.


In [ ]:
training_data = modelling_data[is_training_equipment].copy()
validation_data = modelling_data[is_validation_equipment].copy()
test_data = modelling_data[is_test_equipment].copy()
feature_columns = ["load_pct", "ambient_c"]
print(len(training_data), len(validation_data), len(test_data))


**Example output** (timings and display formatting may vary)

```text
58 15 15
```


## A physical relationship can form a feature
*PowerPoint slide 18*


See the illustration on this PowerPoint slide.

Explain each quantity and the unit of the mean_load_by_equipment_group.


## Derived features from existing inputs
*PowerPoint slide 19*

load_pct / 100 gives load_fraction. Multiplying load_pct by ambient_c gives the load_ambient interaction.

equipment_data["new_name"] = expression adds a column. Apply the same formulas to training and validation rows, without changing raw data.




## Operations for the next example
*PowerPoint slide 20*

Adding or replacing a column: equipment_data["name"] = values assigns that column. An array follows row order. A Series aligns by row label, so keep labels paired with the intended observations.

Column arithmetic: Arithmetic on columns calculates one result per row. Series align by row label. Divide only where the denominator is valid and keep actual and predicted observations paired.




## Defining and calling a function
*PowerPoint slide 21*

def gives a reusable block a name and input parameters. The indented statements run when it is called. return sends a result back to the caller; print displays a result but does not return it.




## Features from the supplied load and ambient columns (1/2)
*PowerPoint slide 22*



Continue in the same code cell.


In [ ]:
def add_features(equipment_data):
    extended_data = equipment_data.copy()
    extended_data["load_fraction"] = (
        extended_data["load_pct"] / 100
    )
    extended_data["load_ambient"] = (
        extended_data["load_pct"] * extended_data["ambient_c"]
    )
    return extended_data


training_features = add_features(training_data)
validation_features = add_features(validation_data)
print(
    training_features[
        ["load_pct", "load_fraction", "load_ambient"]
    ].head()
)


**Example output** (timings and display formatting may vary)

```text
load_pct  load_fraction  load_ambient
0     59.15         0.5915     1875.6465
1     77.46         0.7746     2321.4762
2     69.60         0.6960     1382.9520
6     24.66         0.2466      784.4346
7     72.92         0.7292     2005.3000
```


## Ablation compares the same observations
*PowerPoint slide 24*

Fit one linear model using load_pct and ambient_c. Fit a second using those columns plus load_ambient.

Compare both on the same validation targets. Improvement equals baseline MAE minus extended MAE.




## Operations for the next example
*PowerPoint slide 25*

Rounded display: series.round(3) returns values rounded to three decimal places. Python round(number, 3) rounds one number. Keep unrounded calculations for returned assessment answers.




## A sequence of model steps / Comparing two prediction sets
*PowerPoint slide 26*

make_pipeline(imputer, estimator) links preprocessing and prediction. fit learns the imputer and estimator on training rows. predict applies the learned steps to new rows in the same order.

mean_absolute_error(actual, predicted) averages the absolute differences of paired values. Lower is better. Both lists must have the same length and row order; the answer has the target’s unit.




## Filling model inputs / Creating linear regression
*PowerPoint slide 27*

SimpleImputer(strategy="median") learns a median for each input column during fit. It uses those training medians to fill gaps later. Never learn replacement values from validation or test rows.

LinearRegression() creates an unfitted model. It will learn coefficients and an intercept from training examples. Creating it alone does not train it or make predictions.




## Learning from training examples / Predicting new targets
*PowerPoint slide 28*

model.fit(input_table, observed_targets) learns model settings from paired rows. The two inputs must describe the same observations in the same order. Call fit before predict.

model.predict(input_table) returns one prediction per input row. Use the same feature columns and order as during training. Prediction uses learned settings; it does not refit the model.




## Change one feature set at a time
*PowerPoint slide 29*

The base inputs are load and ambient temperature. The extended inputs also include their product. Train separate pipelines on the same rows and compare on the same validation rows. This isolates the effect of adding the feature.




## Paired ablation using the supplied CSV — step 1
*PowerPoint slide 30*



Compare the printed result with the explanation in the notebook.


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

base_feature_columns = ["load_pct", "ambient_c"]
extended_feature_columns = base_feature_columns + ["load_ambient"]


**Example output** (timings and display formatting may vary)

```text
This cell stores values for the next example; it does not print a result.
```


## Fit both alternatives fairly
*PowerPoint slide 31*

Each pipeline learns its own training medians and regression settings. The base model sees only base columns; the extended model sees the additional feature. Neither uses validation targets while learning.




## Paired ablation using the supplied CSV — step 2 (1/2)
*PowerPoint slide 32*



Continue in the same code cell.


In [ ]:
base_pipeline = make_pipeline(
    SimpleImputer(strategy="median"), LinearRegression()
)
extended_pipeline = make_pipeline(
    SimpleImputer(strategy="median"), LinearRegression()
)
base_pipeline.fit(
    training_features[base_feature_columns],
    training_data["target"],
)
extended_pipeline.fit(
    training_features[extended_feature_columns],
    training_data["target"],
)


**Example output** (timings and display formatting may vary)

```text
This cell stores values for the next example; it does not print a result.
```


## Compare errors on the same rows
*PowerPoint slide 34*

Predict both alternatives on matching validation rows. mean_absolute_error compares each prediction with its observed target. Base MAE minus extended MAE is positive when the extra feature helped on this validation set.




## Paired ablation using the supplied CSV — step 3
*PowerPoint slide 35*



Compare the printed result with the explanation in the notebook.


In [ ]:
base_predictions = base_pipeline.predict(
    validation_features[base_feature_columns]
)
base_absolute_error = mean_absolute_error(
    validation_data["target"], base_predictions
)


**Example output** (timings and display formatting may vary)

```text
This cell stores values for the next example; it does not print a result.
```


## Paired ablation using the supplied CSV — step 4
*PowerPoint slide 36*

extended_predictions: validation predictions from the extended pipeline.

extended_absolute_error: validation mean absolute error for the extended pipeline.




## Paired ablation using the supplied CSV — step 4 (1/2)
*PowerPoint slide 37*



Continue in the same code cell.


In [ ]:
extended_predictions = extended_pipeline.predict(
    validation_features[extended_feature_columns]
)
extended_absolute_error = mean_absolute_error(
    validation_data["target"], extended_predictions
)
print(
    round(base_absolute_error, 4),
    round(extended_absolute_error, 4),
)
print(round(base_absolute_error - extended_absolute_error, 4))


**Example output** (timings and display formatting may vary)

```text
0.7775 0.7992
-0.0217
```


## Pandas inside the required function
*PowerPoint slide 39*

The assignment caller supplies equipment_records. pd.DataFrame(records) gives those records named columns. Calculate with pandas, then return the published Python type.

to_dict("records") makes a list of row dictionaries. Series.tolist() makes a list of values. A grouped Series.to_dict() maps group labels to validation_results.




## Operations for the next example
*PowerPoint slide 40*

Returning a dictionary: series.to_dict() maps Series labels to values. equipment_data.to_dict("records") instead returns a list of row dictionaries. Choose the form required by the assignment.

A table from Python values: pd.DataFrame(records) builds a table from row dictionaries. With a dictionary of column lists, keys become column names and each list supplies that column’s values.

Mean of recorded values: series.mean() adds known values and divides by their count. pandas skips missing values by default. The result has the same unit as the original column.




## Operations for the next example
*PowerPoint slide 41*

Grouping rows: equipment_data.groupby("label")["value"] groups the value column by matching labels. Apply mean() or sum() to calculate one result per group. Missing group labels are excluded by default.




## A function called with the supplied records
*PowerPoint slide 42*



Compare the printed result with the explanation in the notebook.


In [ ]:
def mean_load_by_equipment(equipment_records):
    equipment_data = pd.DataFrame(equipment_records)
    mean_load_by_equipment_group = equipment_data.groupby(
        "equipment_id"
    )["load_pct"].mean()
    return mean_load_by_equipment_group.to_dict()


equipment_records = raw_data.to_dict("records")
print(mean_load_by_equipment(equipment_records))


**Example output** (timings and display formatting may vary)

```text
{'E1': 54.9475, 'E2': 61.326, 'E3': 53.46933333333333, 'E4': 66.854, 'E5': 48.10133333333333, 'E6': 51.54}
```


## Continue with your issued assignment
Open the assignment notebook named in 00_START_HERE.txt. Keep required function names, return types and units. guided_practice.ipynb is not a graded upload. Use SUBMIT_HERE.txt for the correct submission link.
